# Bước 3: Tối ưu & Cải tiến hiệu năng

**Mục tiêu:** xử lý mất cân bằng dữ liệu bằng SMOTE và hướng tới mốc hiệu
năng cao của `ishaq2021improving` — Ishaq et al. (2021), *IEEE Access* —
Extra Trees + SMOTE, Accuracy 92,6%.

::: danger Cảnh báo đỏ (Red Flag) — đúng như đề bài gốc
SMOTE tuyệt đối **chỉ được áp dụng trên tập Train**. Nếu SMOTE cả tập Test,
mô hình được đánh giá trên dữ liệu tổng hợp (ảo) thay vì bệnh nhân thật —
làm sai lệch hoàn toàn ý nghĩa lâm sàng của kết quả.

In [1]:
import sys
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

try:
    import imblearn  # noqa: F401
except ImportError:
    import subprocess, sys as _sys
    print("[i] Chưa có imbalanced-learn trong môi trường này (Colab/Kaggle) — đang cài...")
    subprocess.run([_sys.executable, '-m', 'pip', 'install', '-q', 'imbalanced-learn'], check=True)

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import accuracy_score, matthews_corrcoef, f1_score, classification_report
from imblearn.over_sampling import SMOTE

def _load_heart_failure_data():
    """Tải dữ liệu cục bộ (../data/...) nếu có (chạy trong repo HMYT đã
    clone); nếu không (mở độc lập qua Colab/Kaggle, không có thư mục data/
    đi kèm) tự động tải từ mirror công khai trên hmyt-book (repo Public,
    xác minh 23/09/2026)."""
    import os
    local_path = '../data/heart_failure_clinical_records_dataset.csv'
    remote_url = ('https://raw.githubusercontent.com/fossbk-spec/hmyt-book/gh-pages/'
                  'labs_chuyen_de/ch02_suy_tim_risk_dxai/data/'
                  'heart_failure_clinical_records_dataset.csv')
    path = local_path if os.path.exists(local_path) else remote_url
    if path == remote_url:
        print(f"[i] Không tìm thấy dữ liệu cục bộ — tự động tải từ mirror công khai:\n    {remote_url}")
    return pd.read_csv(path).rename(columns={'death_event': 'DEATH_EVENT'})

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

df = _load_heart_failure_data()
FEATURE_COLS = [c for c in df.columns if c != 'DEATH_EVENT']
X, y = df[FEATURE_COLS], df['DEATH_EVENT'].values

# Giữ NGUYÊN cách chia Train/Test như Bước 1-2 để so sánh công bằng
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)   # Test giữ NGUYÊN BẢN, KHÔNG qua SMOTE

print(f"Trước SMOTE — Train: {len(y_train)} mẫu, tỷ lệ tử vong {y_train.mean()*100:.1f}%")

Trước SMOTE — Train: 239 mẫu, tỷ lệ tử vong 32.2%


## 1. Áp dụng SMOTE — CHỈ trên tập Train

In [2]:
smote = SMOTE(random_state=RANDOM_STATE)
X_train_smote, y_train_smote = smote.fit_resample(X_train_s, y_train)

print(f"Sau SMOTE — Train: {len(y_train_smote)} mẫu, tỷ lệ tử vong {y_train_smote.mean()*100:.1f}%")
print(f"(Đã cân bằng 2 lớp: {np.sum(y_train_smote==0)} sống sót / {np.sum(y_train_smote==1)} tử vong)")
print(f"\nTập Test KHÔNG đổi: {len(y_test)} mẫu, tỷ lệ tử vong {y_test.mean()*100:.1f}% (giữ nguyên phân bố thật)")

Sau SMOTE — Train: 324 mẫu, tỷ lệ tử vong 50.0%
(Đã cân bằng 2 lớp: 162 sống sót / 162 tử vong)

Tập Test KHÔNG đổi: 60 mẫu, tỷ lệ tử vong 31.7% (giữ nguyên phân bố thật)


## 2. Huấn luyện Extra Trees trên dữ liệu đã SMOTE

In [3]:
etc = ExtraTreesClassifier(n_estimators=300, random_state=RANDOM_STATE)
etc.fit(X_train_smote, y_train_smote)
y_pred_etc = etc.predict(X_test_s)   # Đánh giá trên Test THẬT, không qua SMOTE

acc_etc = accuracy_score(y_test, y_pred_etc)
mcc_etc = matthews_corrcoef(y_test, y_pred_etc)
f1_etc = f1_score(y_test, y_pred_etc)
print(f"Extra Trees + SMOTE -> Accuracy: {acc_etc:.4f} | MCC: {mcc_etc:.4f} | F1: {f1_etc:.4f}")
print()
print(classification_report(y_test, y_pred_etc, target_names=['Sống sót', 'Tử vong']))

Extra Trees + SMOTE -> Accuracy: 0.7667 | MCC: 0.4182 | F1: 0.5333

              precision    recall  f1-score   support

    Sống sót       0.78      0.93      0.84        41
     Tử vong       0.73      0.42      0.53        19

    accuracy                           0.77        60
   macro avg       0.75      0.67      0.69        60
weighted avg       0.76      0.77      0.75        60



## 3. So sánh Có SMOTE vs. Không SMOTE (cùng thuật toán Extra Trees)

In [4]:
etc_no_smote = ExtraTreesClassifier(n_estimators=300, random_state=RANDOM_STATE)
etc_no_smote.fit(X_train_s, y_train)   # Không SMOTE, huấn luyện trực tiếp trên Train mất cân bằng
y_pred_no_smote = etc_no_smote.predict(X_test_s)

acc_no = accuracy_score(y_test, y_pred_no_smote)
mcc_no = matthews_corrcoef(y_test, y_pred_no_smote)
f1_no = f1_score(y_test, y_pred_no_smote)

print("=== BẢNG SO SÁNH CÓ SMOTE vs. KHÔNG SMOTE (cùng Extra Trees, cùng Test) ===")
print(f"{'Cấu hình':<28}{'Accuracy':>12}{'MCC':>10}{'F1':>10}")
print(f"{'Extra Trees (không SMOTE)':<28}{acc_no:>12.4f}{mcc_no:>10.4f}{f1_no:>10.4f}")
print(f"{'Extra Trees + SMOTE':<28}{acc_etc:>12.4f}{mcc_etc:>10.4f}{f1_etc:>10.4f}")
print(f"{'ishaq2021improving (y văn)':<28}{'0.9260':>12}{'-':>10}{'-':>10}")
print(f"\nChênh lệch Accuracy so với y văn: {acc_etc - 0.926:+.4f}")

=== BẢNG SO SÁNH CÓ SMOTE vs. KHÔNG SMOTE (cùng Extra Trees, cùng Test) ===
Cấu hình                        Accuracy       MCC        F1
Extra Trees (không SMOTE)         0.7500    0.3685    0.4828
Extra Trees + SMOTE               0.7667    0.4182    0.5333
ishaq2021improving (y văn)        0.9260         -         -

Chênh lệch Accuracy so với y văn: -0.1593


## 4. Tổng kết Bước 3

Số liệu ở 2 cell trên là kết quả **thực chạy**, không gán cứng theo mốc y
văn. Nếu Accuracy chưa tiệm cận 92,6%, các hướng cải thiện hợp lệ (không
đổi phạm vi Test, không SMOTE Test): tăng `n_estimators`, thử
`RandomForestClassifier`/`GradientBoostingClassifier` thay Extra Trees,
hoặc kết hợp `SMOTETomek`/`SMOTEENN` (lọc nhiễu biên sau khi sinh mẫu) —
đúng tinh thần "Ablation Study rút gọn" của đề bài gốc.

**Tiếp theo:** [`4_xai.ipynb`](./4_xai.ipynb) — Bước 4, giải thích mô hình
bằng SHAP và đối chiếu Serum Creatinine / Ejection Fraction.